## Environment Variables Setup (Required)

### Option: Use a `.env` file (Recommended)
Create a `.env` file in your project root:

```env
DATABRICKS_HOST=https://your-workspace.azuredatabricks.net
DATABRICKS_TOKEN=your_personal_access_token_here
```

Then ensure your notebook/script loads `.env` using `load_dotenv()` before creating the client.


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads DATABRICKS_HOST and DATABRICKS_TOKEN from .env
# Verify your variables loaded correctly:
print("DATABRICKS_HOST:", "SET" if os.getenv("DATABRICKS_HOST") else "NOT SET")
print("DATABRICKS_TOKEN:", "SET" if os.getenv("DATABRICKS_TOKEN") else "NOT SET")

DATABRICKS_HOST: SET
DATABRICKS_TOKEN: SET


### 1. Development using LLM Native API

In [4]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

client = WorkspaceClient()

topic = "LangChain"
response = client.serving_endpoints.query(
    name="databricks-meta-llama-3-3-70b-instruct",
    messages=[ChatMessage(role=ChatMessageRole.USER, content=f"Explain {topic} in 3 short bullet points.")]
)

result = response.choices[0].message.content
print(result)

- LangChain is a framework designed to simplify building applications with large language models (LLMs) by enabling easy integration of various components like prompts, memory, and chains.  
- It supports chaining together multiple LLM calls and tools to create complex workflows and applications such as chatbots, question-answering systems, and agents.  
- LangChain provides modular abstractions for prompt management, document retrieval, and interaction with external data sources, enhancing the capabilities and flexibility of LLM-powered apps.


### 2. Developing using Langchain API

In [5]:
from databricks_langchain import ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0,
    max_tokens=500
)

# Basic Example 1: call the chat model directly
msg = llm.invoke("What is Python in one sentence?")
print("Example 1:", msg.content)

Example 1: Python is a high-level, interpreted programming language known for its simplicity, readability, and versatility, making it a popular choice for various applications, including web development, data analysis, artificial intelligence, and more.


### 3. Creating Basic Chain using Langchain Expression Language (LCEL)

In [6]:
# Basic Example 2: simple prompt + parser chain
prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in 3 short bullet points."
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "LangChain"})
print(result)

- LangChain is a framework designed to simplify building applications with large language models (LLMs) by chaining together various components like prompts, models, and memory.
- It enables developers to create complex workflows that combine LLMs with external data sources, APIs, and state management for enhanced functionality.
- LangChain supports integrations with multiple LLM providers and tools, facilitating modular and scalable development of AI-powered applications.


In [7]:
# Basic Example 3: reuse your existing chain with different topics
for t in ["LangChain", "Prompt Engineering", "Vector Databases"]:
    print(f"Example 3 ({t}):")
    print(chain.invoke({"topic": t}))
    print("-" * 60)

Example 3 (LangChain):
- LangChain is a framework designed to simplify building applications with large language models (LLMs) by chaining together components like prompts, models, and memory.  
- It enables developers to create complex workflows involving text generation, retrieval, and interaction with external data sources.  
- LangChain supports integrations with various APIs and tools, facilitating the development of intelligent, context-aware applications.
------------------------------------------------------------
Example 3 (Prompt Engineering):
- Designing clear and specific inputs to guide AI models in generating desired outputs.  
- Iteratively refining prompts to improve accuracy, relevance, and creativity of responses.  
- Leveraging understanding of model behavior to optimize performance for various tasks.
------------------------------------------------------------
Example 3 (Vector Databases):
- **Purpose:** Vector databases store and manage high-dimensional vector embe